<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">🛡️ Lab 06: Red-Team Your LangGraph Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Final security gate — confirm that security posture depends on model and instructions, not framework</p>
</div>

**What We Learn:** Red teaming confirms that security posture depends on the model and instructions, not the framework. Same attacks, same API, potentially different results based on how instructions are structured.

---


## 🧳 The Contoso Travel Story

Final security gate for the entire workshop — validating that the agent is production-ready regardless of implementation.

- **The Problem:** Three implementations, potentially three different vulnerability profiles. The team needs to confirm that security posture is consistent — or understand why it differs.
- **This Lab Solves:** Running the same red team scan to complete the three-way comparison and confirm that security is an API-level concern.

## 1. Setup

---


In [ ]:
import os
import time
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    RedTeam,
    AzureOpenAIModelConfiguration,
    AttackStrategy,
    RiskCategory,
)

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")
model_endpoint = os.environ["MODEL_ENDPOINT"]
model_api_key = os.environ["MODEL_API_KEY"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Microsoft Foundry")
print(f"   Model: {model_name}")
print(f"   Model Endpoint: {model_endpoint[:30]}...")

## 2. Agent-Agnostic Red Teaming

Red team attacks go through the **Responses API**. The red teaming service sends adversarial prompts and evaluates responses — it has no knowledge of `BaseAgent`, `StateGraph`, or `PromptAgentDefinition`.

What determines how well the agent resists attacks:
- The **model** itself (its built-in safety training)
- The **system instructions** (guardrails you define)
- The **content filters** configured at the Azure OpenAI level

What does *not* matter:
- Whether the agent is a prompted agent, MAF agent, or LangGraph graph
- The framework's internal architecture

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🔁 Same scan, third time:</b> If the results differ from the Prompted or MAF tracks, the difference comes from the model configuration or instructions — not the framework.
</div>

---


## 3. Configure Red Team Scan

---


In [ ]:
red_team = RedTeam(
    attack_strategies=[AttackStrategy.JAILBREAK, AttackStrategy.CRESCENDO],
    risk_categories=[RiskCategory.VIOLENCE, RiskCategory.HATE_UNFAIRNESS],
    display_name="contoso-travel-langgraph-redteam",
    target=AzureOpenAIModelConfiguration(model_deployment_name=model_name),
    num_objectives=5,
)

print("✅ Red Team configuration created")
print(f"   Risk categories: Violence, Hate/Unfairness")
print(f"   Attack strategies: Jailbreak, Crescendo")
print(f"   Objectives: 5")
print(f"   Target model: {model_name}")

## 4. Run the Scan

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⏱️ Note:</b> Red team scans can take several minutes to complete depending on the number of risk categories and attack strategies.
</div>

---


In [ ]:
print("🚀 Starting Red Team scan...")

red_team_response = project_client.beta.red_teams.create(
    red_team=red_team,
    headers={
        "model-endpoint": model_endpoint,
        "model-api-key": model_api_key,
    },
)

print(f"✅ Red Team scan created!")
print(f"   Scan name: {red_team_response.name}")
print(f"   Status: {red_team_response.status}")

# Monitor scan progress
scan_name = red_team_response.name

while True:
    scan_status = project_client.beta.red_teams.get(name=scan_name)
    print(f"   ⏳ Status: {scan_status.status}")

    if scan_status.status in ["Completed", "Failed", "completed", "failed"]:
        break
    time.sleep(15)

print(f"\n{'✅' if 'omplete' in str(scan_status.status) else '❌'} Scan {scan_status.status}!")

## 5. Review Results

---


In [ ]:
print(f"📊 Red Team Scan Results")
print(f"{'='*60}")
print(f"   Scan Name: {scan_name}")
print(f"   Status: {scan_status.status}")

# List all scans for context
print(f"\n📋 All Red Team Scans:")
for scan in project_client.beta.red_teams.list():
    print(f"   • {scan.name} — Status: {scan.status}")

print(f"\n💡 For detailed results, visit the Foundry portal:")
print(f"   1. Go to https://ai.azure.com")
print(f"   2. Open your project → Evaluations → Red Teaming")
print(f"   3. Review individual attack attempts and model responses")

## 6. Three-Way Comparison

If you've completed Labs 06 across all three tracks, compare your red team results:

| Risk Category | Prompted | MAF | LangGraph |
|---------------|----------|-----|-----------|
| Violence | ___ | ___ | ___ |
| Hate/Unfairness | ___ | ___ | ___ |

| Attack Strategy | Prompted | MAF | LangGraph |
|-----------------|----------|-----|-----------|
| Jailbreak | ___ | ___ | ___ |
| Crescendo | ___ | ___ | ___ |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Expected result:</b> Security posture should be similar across tracks because attacks target the model and instructions, not the framework. Any differences point to instruction tuning, not architecture.
</div>

---


## 7. Workshop Wrap-Up

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">🎉 Congratulations! You've completed the full journey.</h3>
  <p>Across three implementation tracks, you've built, traced, evaluated, and red-teamed AI agents for the Contoso Travel platform:</p>
  <ul style="margin-bottom: 0;">
  <li><b>Prompted Agents</b> — Direct model interaction with <code>PromptAgentDefinition</code></li>
  <li><b>MAF Agents</b> — Custom agent logic with <code>BaseAgent</code> and the Multi-Agent Framework</li>
  <li><b>LangGraph Agents</b> — Graph-based orchestration with <code>StateGraph</code></li>
  </ul>
  <p style="margin-bottom: 0; margin-top: 12px;">And at every step, you proved that <b>observability, evaluation, and security are agent-agnostic</b> — the same tools work regardless of how the agent is built.</p>
</div>

### What You Accomplished Across All Tracks

| Capability | Lab | What You Proved |
|------------|-----|----------------|
| **Tracing** | 04 | OpenTelemetry works with any agent — flat spans, custom spans, or graph-topology spans |
| **Evaluation** | 05 | Quality and safety evaluators are API-level — same code, same results |
| **Red Teaming** | 06 | Security posture depends on model + instructions, not framework |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🧪 Your sandbox:</b> This repository is now your sandbox — try new attack strategies, custom evaluators, different models, or extend the agents with new capabilities.
</div>

---


## 8. Cleanup

---


In [ ]:
project_client.close()
credential.close()
print("✅ Clients closed")

## 9. Summary

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Ran <b>red team scans</b> with Jailbreak and Crescendo attack strategies</li>
  <li>Tested against Violence and Hate/Unfairness risk categories</li>
  <li>Confirmed that <b>security is an API-level concern</b> — framework choice doesn't affect vulnerability profile</li>
  <li>Completed the <b>three-way comparison</b> across Prompted, MAF, and LangGraph tracks</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <b>💡 Key Takeaway:</b> Red teaming is the final safety gate before production. It confirms that your agent resists adversarial attacks — and the results depend on the model and instructions, not the framework used to build the agent.
</div>

---
